<a href="https://colab.research.google.com/github/HemanthB-007/Project-Retention/blob/main/Project_Retention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install XGBoost if you haven't already: !pip install xgboost scikit-learn pandas numpy matplotlib seaborn
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

print("[*] Initializing Project Retention: Customer Churn Engine...")

# 1. Generate a realistic Synthetic Customer Dataset (10,000 users)
np.random.seed(42)
n_samples = 10000

data = {
    'Tenure_Months': np.random.randint(1, 72, n_samples),
    'Monthly_Charges': np.random.uniform(20.0, 120.0, n_samples),
    'Support_Calls': np.random.randint(0, 10, n_samples),
    'Payment_Delay_Days': np.random.randint(0, 30, n_samples),
    'Usage_Frequency': np.random.randint(1, 100, n_samples),
    'Contract_Type': np.random.choice([0,1,2], n_samples), # 0: M2M, 1: 1-Year, 2: 2-Year
}

df = pd.DataFrame(data)

# Formulate the Churn Target (Logic: High calls, high delays, low tenure = higher churn risk)
churn_prob = (df['Support_Calls'] * 0.1) + (df['Payment_Delay_Days'] * 0.05) - (df['Tenure_Months'] * 0.01) - (df['Contract_Type'] * 0.2)
# Convert probability to binary 0 (Stayed) or 1 (Churned)
df['Churn'] = (churn_prob > np.percentile(churn_prob, 75)).astype(int)

print(f"[+] Dataset Generated: {df.shape[0]} customers, {df.shape[1]-1} features.")
print(f"[+] Class Distribution:\n{df['Churn'].value_counts(normalize=True) * 100}")

[*] Initializing Project Retention: Customer Churn Engine...
[+] Dataset Generated: 10000 customers, 6 features.
[+] Class Distribution:
Churn
0    75.05
1    24.95
Name: proportion, dtype: float64


In [ ]:
# 2. Preprocessing
X = df.drop('Churn', axis=1)
y = df['Churn']

# Split into Training (80%) and Testing (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Scale the features (Crucial for Logistic Regression)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\n[*] Training AI Models...")

# 3. Initialize Models
models = {
    "Logistic Regression": LogisticRegression(class_weight='balanced'),
    "Random Forest": RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42),
    "XGBoost Champion": xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss', scale_pos_weight=3, random_state=42)
}

results = {}

# Train and Evaluate each model
for name, model in models.items():
    # Use scaled data for LR, raw data for trees
    X_train_use = X_train_scaled if name == "Logistic Regression" else X_train
    X_test_use = X_test_scaled if name == "Logistic Regression" else X_test

    model.fit(X_train_use, y_train)
    y_pred = model.predict(X_test_use)

    # Calculate Metrics
    results[name] = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1-Score": f1_score(y_test, y_pred)
    }

print("[+] Training Complete.")


[*] Training AI Models...
[+] Training Complete.


In [ ]:
# 4. Display the Benchmark Results
print("\n" + "="*50)
print("🏆 MODEL EVALUATION BENCHMARKS 🏆")
print("="*50)

results_df = pd.DataFrame(results).T
print(results_df.round(4) * 100) # Display as percentages

# 5. Extract Actionable Insights (Feature Importance from XGBoost)
print("\n" + "="*50)
print("🧠 XGBoost Feature Importance (What drives churn?)")
print("="*50)

xgb_model = models["XGBoost Champion"]
importances = xgb_model.feature_importances_
feature_names = X.columns

# Sort and display
feat_imp = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
feat_imp = feat_imp.sort_values(by='Importance', ascending=False)
print(feat_imp)


🏆 MODEL EVALUATION BENCHMARKS 🏆
                     Accuracy  Precision  Recall  F1-Score
Logistic Regression     99.00      96.15  100.00     98.04
Random Forest           97.35      96.27   92.99     94.60
XGBoost Champion        97.60      94.13   96.39     95.25

🧠 XGBoost Feature Importance (What drives churn?)
              Feature  Importance
3  Payment_Delay_Days    0.428822
2       Support_Calls    0.217021
5       Contract_Type    0.194014
0       Tenure_Months    0.138216
4     Usage_Frequency    0.011727
1     Monthly_Charges    0.010202
